# 00 · Colab setup and data caches

Run top to bottom **once** to build the preprocessed caches on Google Drive
(`docs/preprocessing_v1.md`). Later notebooks only mount Drive and load the caches.

Runtime: GPU is **not** needed for this notebook (Runtime → Change runtime type → CPU is fine).
Expected time: PTB-XL download ~5–10 min, PTB-XL cache ~15–25 min, SPH ~10–20 min.
Every step is resumable: if the session dies, re-run from the top; finished steps are skipped.

**Rules for Colab:**
- Use **Runtime → Run all**. If the runtime restarts, run all again (finished steps are skipped).
- **Do not use File → Save a copy in GitHub.** It overwrites the repo's notebook with your session's
  copy, outputs included. If you want a record of a run, use File → Save a copy in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/robust-ecg-ssl'   # persistent: caches, checkpoints, results
CACHE   = f'{PROJECT}/cache'
RAW     = f'{PROJECT}/raw'                          # downloaded archives (kept so you download once)
LOCAL   = '/content/data'                           # fast local disk, wiped each session
for p in (CACHE, RAW, LOCAL): os.makedirs(p, exist_ok=True)

In [ ]:
# Code: always pull the latest version of the repo
REPO = '/content/robust-ecg-ssl'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Salako07/robust-ecg-ssl.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
!pip -q install wfdb h5py

## PTB-XL v1.0.3 (PhysioNet, CC BY 4.0)
If the URL below returns 404, copy the "Download the ZIP file" link from
https://physionet.org/content/ptb-xl/1.0.3/ and paste it into `PTBXL_URL`.

In [ ]:
PTBXL_URL = 'https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip'
zip_path = f'{RAW}/ptbxl-1.0.3.zip'
if not os.path.exists(f'{CACHE}/ptbxl_X.npy'):
    if not os.path.exists(zip_path):
        !wget -q --show-progress -O {zip_path} "{PTBXL_URL}"
    !unzip -q -n {zip_path} -d {LOCAL}/ptbxl_zip
    PTBXL_DIR = [os.path.join(r) for r, d, f in os.walk(f'{LOCAL}/ptbxl_zip') if 'ptbxl_database.csv' in f][0]
    print('PTB-XL folder:', PTBXL_DIR)
    !python {REPO}/scripts/prepare_ptbxl.py --ptbxl-dir "{PTBXL_DIR}" --out {CACHE}
else:
    print('PTB-XL cache already built')

## SPH (Shandong Provincial Hospital, figshare, CC BY 4.0)
Collection: https://doi.org/10.6084/m9.figshare.c.5779802 . The cell lists every file in the
collection and downloads them to Drive. **Check the printed file list**: the preparation step expects
`metadata.csv` and a `records/` folder of `.h5` files after extraction. If the layout differs, adjust
`SPH_DIR` in the next cell.

In [ ]:
import json, urllib.request
def get(url): return json.load(urllib.request.urlopen(url))
sph_raw = f'{RAW}/sph'; os.makedirs(sph_raw, exist_ok=True)
for art in get('https://api.figshare.com/v2/collections/5779802/articles?page_size=100'):
    for f in get(f"https://api.figshare.com/v2/articles/{art['id']}")['files']:
        dest = f"{sph_raw}/{f['name']}"
        print(f"{f['name']:40s} {f['size']/1e6:9.1f} MB", '(have)' if os.path.exists(dest) else '')
        if not os.path.exists(dest):
            !wget -q -O "{dest}" "{f['download_url']}"
!ls -la {sph_raw}

In [ ]:
# Extract archives to local disk, then locate metadata.csv and the .h5 records
import glob
os.makedirs(f'{LOCAL}/sph', exist_ok=True)
import shutil, subprocess
for a in glob.glob(f'{sph_raw}/*'):
    if a.endswith('.zip'):
        subprocess.run(['unzip', '-q', '-n', a, '-d', f'{LOCAL}/sph'], check=True)
    elif a.endswith(('.tar.gz', '.tgz', '.tar')):
        subprocess.run(['tar', '-xf', a, '-C', f'{LOCAL}/sph'], check=True)
    else:
        shutil.copy(a, f'{LOCAL}/sph/')
h5 = glob.glob(f'{LOCAL}/sph/**/*.h5', recursive=True)
metas = glob.glob(f'{LOCAL}/sph/**/metadata.csv', recursive=True)
print(len(h5), 'h5 files;', 'metadata:', metas)
SPH_DIR = f'{LOCAL}/sph_std'                  # standard layout expected by the scripts
os.makedirs(f'{SPH_DIR}/records', exist_ok=True)
!cp -n "{metas[0]}" {SPH_DIR}/metadata.csv
for p in h5:
    t = f'{SPH_DIR}/records/{os.path.basename(p)}'
    if not os.path.exists(t): os.symlink(p, t)
print('records linked:', len(os.listdir(f'{SPH_DIR}/records')))

In [ ]:
# Deduplicate (D14), then build the SPH cache. Each step stops the cell if it fails.
import subprocess
def run(*args):
    subprocess.run(['python', *args], check=True)
if not os.path.exists(f'{CACHE}/sph_X.npy'):
    run(f'{REPO}/scripts/sph_dedup.py', '--sph-dir', f'{SPH_DIR}/records',
        '--meta', f'{SPH_DIR}/metadata.csv', '--out-dir', f'{PROJECT}/results')
    run(f'{REPO}/scripts/prepare_sph.py', '--sph-dir', SPH_DIR,
        '--exclude', f'{PROJECT}/results/sph_exclude.txt', '--out', CACHE)
else:
    print('SPH cache already built')

## Check before moving on
Both amplitude reports should show QRS-scale values in **mV** (per-lead p99 of max |x| roughly
0.5–3 mV) and zero non-finite records. **Copy the printed dedup counts and both amplitude reports
into the chat** so they can be recorded in the decision log (D14) and preprocessing doc.

In [ ]:
!ls -la {CACHE} {PROJECT}/results